# 08 · Orchestration patterns

**Course:** Building LLM Agents, notebook 8 of 10
**Where we are:** in notebook 07 you had two agents talk to each other, a specialist handing work to another specialist. That was one connection. Real systems have many agents, and the interesting question becomes: how do you wire them together?

### What you'll learn
The four named patterns for coordinating many agents, each with a tiny, runnable implementation:
1. Sequential pipeline (chain): agents in a fixed order, each transforms the previous output
2. Router / dispatcher: a classifier sends each request to the right specialist
3. Orchestrator-worker: a manager decomposes a task, workers solve the pieces, the manager synthesises
4. Parallel fan-out / map-reduce: run independent workers concurrently, then aggregate

### Prerequisites
- Notebooks 01 to 07: chat calls, JSON mode, tools, the agent loop, memory, a robust single agent, and two agents passing messages
- Your `.env` is set up and `from helpers import ...` works

> Mental model: an orchestration pattern is a shape of data flow between agents. You already know how to make one agent produce an output; a pattern is the plumbing that decides who runs, in what order, and who reads whose output. There are only a handful of fundamental shapes, and almost every multi-agent system you'll ever see is one of them, or a composition of them.

> Recap of 07. You built two specialised agents (each = model + role + tools) and had the output of one become the input of the next, a manual handoff. This notebook generalises that single handoff into reusable coordination patterns for N agents. The building block is unchanged: an agent is still just a thing that takes a task string and returns an answer string.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import json
import time

from helpers import client, MODEL, ask, jprint

For these patterns we mostly use the one-shot helper `ask(prompt, system=..., **kwargs)` from `helpers.py`. It sends a single prompt and returns the text answer, which suits us when we're firing off many small calls and the coordination is what matters, not the mechanics of one call.

> What counts as "an agent" here. To keep the focus on the patterns, each of our agents is a small function that takes text and returns text. That is deliberately the simplest thing that fits the shape. Everything below works identically if you swap in a full `Agent` (from notebook 04) with its own tools and loop. The wiring is the same.

Here is a trivially small "agent" so the shapes are easy to see.

In [ ]:
def make_agent(name, system, **kwargs):
    """Return a tiny agent: a function task->answer with a fixed role."""
    def agent(task):
        return ask(task, system=system, temperature=0, **kwargs)
    agent.__name__ = name
    return agent

# A quick sanity check that our agent-as-a-function works.
hello = make_agent("greeter", "You are friendly. Reply in under 8 words.")
print(hello("Say hi to the class."))

![The four orchestration patterns side by side](figures/08-four-patterns.svg)

*Figure: pipeline, router, orchestrator-worker, and parallel fan-out, the shapes of "who runs when". Generate it from the prompt in [`figures/FIGURES.md`](figures/FIGURES.md).*

## 1. Sequential pipeline (chain)

The simplest pattern. Agents run in a fixed order, and each one transforms the output of the one before it. No branching, no decisions, just a conveyor belt.

```
task ─▶ [ Agent A ] ─▶ [ Agent B ] ─▶ [ Agent C ] ─▶ result
         outline         draft          polish
```

**When to use it:** when the work has natural stages that always happen in the same order: outline -> draft -> polish, extract -> transform -> format, translate -> summarise. If you can describe the job as "first do X, then do Y to the result," it's a pipeline.

The implementation is almost embarrassingly small: fold the task through a list of agents.

In [ ]:
def pipeline(agents, task, verbose=True):
    """Run agents in order; each one transforms the previous output."""
    data = task
    for agent in agents:
        data = agent(data)
        if verbose:
            print(f"  ↳ after {agent.__name__:8}: {data[:70]!r}")
    return data

Now three stage-agents for a writing task. Notice each one's role is tightly scoped to its job. That is the whole point of decomposing into stages.

In [ ]:
outliner = make_agent(
    "outline",
    "You are an outliner. Return a 2-bullet outline. Nothing else.",
    max_tokens=8000,
)
drafter = make_agent(
    "draft",
    "You turn an outline into 2 short sentences of prose.",
    max_tokens=8000,
)
polisher = make_agent(
    "polish",
    "You are an editor. Tighten the text into one punchy sentence.",
    max_tokens=8000,
)

final = pipeline([outliner, drafter, polisher], "the benefits of cycling to work")
print("\nFINAL:", final)

> Key idea: each stage only has to be good at one narrow thing, and you can test, swap, or reorder stages independently. That modularity is why pipelines are the backbone of so many LLM systems.

> Gotcha: errors propagate. If the outliner returns junk, the drafter faithfully drafts junk. In production you'd validate the output of each stage before passing it on (notebook 06's guardrails apply per stage).

## 2. Router / dispatcher

A pipeline always runs every stage. But often you want to send a request to the right specialist and skip the rest. A router is a lightweight classifier, one cheap LLM call that returns a category, followed by a dispatch to the matching agent.

```
                       ┌─▶ [ math agent ]
request ─▶ [ router ] ─┼─▶ [ coding agent ]
           (classify)  └─▶ [ general agent ]
```

**When to use it:** when you have several specialists and each request belongs to one of them: a support bot that triages "billing vs. technical vs. sales," a coding assistant that splits "explain vs. write vs. debug." One classify, then one specialist runs.

> Rule of thumb: make the router constrained and cheap. Ask for a single category, use JSON mode so the answer is machine-readable, set `temperature=0` (this is a decision, not creativity), and cap `max_tokens` low. The router doesn't solve the problem; it just points at who should.

In [ ]:
# Our pool of specialists. Keep replies short to limit tokens.
SPECIALISTS = {
    "math":    make_agent("math",    "You are a maths expert. Reply with the number only.",          max_tokens=8000),
    "coding":  make_agent("coding",  "You are a coding expert. Reply with one line of code only.",    max_tokens=8000),
    "general": make_agent("general", "You are a helpful generalist. Answer in one short sentence.",   max_tokens=8000),
}
CATEGORIES = list(SPECIALISTS)  # ["math", "coding", "general"]
print("specialists:", CATEGORIES)

The classifier uses `response_format={"type": "json_object"}` (the clean JSON mode you met in notebook 02) so we get back something we can `json.loads` without fragile string-parsing. We also defend against a surprise category by falling back to `general`.

In [ ]:
def classify(request):
    """One cheap, constrained LLM call -> a category string."""
    system = (
        "Classify the user's request into EXACTLY one category: "
        f"{CATEGORIES}. "
        'Reply ONLY as JSON: {"category": "<one of the categories>"}.'
    )
    raw = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": request},
        ],
        response_format={"type": "json_object"},  # clean JSON, no prose
        temperature=0,                              # a decision, so be deterministic
        max_tokens=8000,
    ).choices[0].message.content
    category = json.loads(raw).get("category", "general")
    return category if category in SPECIALISTS else "general"  # defensive fallback

In [ ]:
def route(request):
    """Classify, then dispatch to the chosen specialist."""
    category = classify(request)
    answer = SPECIALISTS[category](request)
    return category, answer

Now send three different kinds of request through the same `route()` and watch them land on three different specialists.

In [ ]:
requests = [
    "What is 17 * 23?",
    "Write a Python one-liner to reverse a string.",
    "Name a country in Scandinavia.",
]

for req in requests:
    category, answer = route(req)
    print(f"[{category:7}] {req}")
    print(f"          → {answer}\n")

> Why you care. The router is your traffic cop. It keeps each specialist's prompt small and focused (cheaper, more reliable) instead of stuffing one mega-prompt with "if it's maths do this, if it's code do that...". And because the routing decision is just another LLM call, you can make it as smart as you need, or replace it with plain keyword rules when the categories are obvious.

> Gotcha: a router is only as good as its categories. If a request fits two of them, or none, the classifier has to guess. Keep categories distinct, give the model a `general` escape hatch, and validate the returned label (we did, with the fallback).

## 3. Orchestrator-worker (manager decomposes)

In a pipeline you hard-code the stages. In a router you hard-code the specialists. But sometimes you don't know the steps in advance: they depend on the task. The orchestrator-worker pattern lets an LLM plan the steps itself: a manager agent breaks the task into subtasks, worker agents solve each one, and the manager synthesises the results.

```
                      ┌─▶ [ worker ] ─┐
task ─▶ [ orchestrator ] ─▶ [ worker ] ─┼─▶ [ orchestrator ] ─▶ result
        decompose (JSON)  └─▶ [ worker ] ─┘    synthesise
```

**When to use it:** open-ended tasks where the right breakdown isn't known ahead of time, such as "research X," "plan an event," or "review this document from three angles." The manager adapts the plan to the specific request.

> Mental model: this is "manager + team." The manager doesn't do the work; it decides what the work is, delegates, and writes up the result. The decomposition step is the new idea, and we get it for free by asking the model for a JSON list of subtasks.

In [ ]:
def decompose(task, max_subtasks=3):
    """Ask the manager to split a task into a JSON list of subtasks."""
    system = (
        f"You are a planner. Break the task into at most {max_subtasks} "
        "small, INDEPENDENT subtasks that can each be answered on their own. "
        'Reply ONLY as JSON: {"subtasks": ["...", "..."]}.'
    )
    raw = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": task},
        ],
        response_format={"type": "json_object"},
        temperature=0,
        max_tokens=8000,
    ).choices[0].message.content
    return json.loads(raw).get("subtasks", [])[:max_subtasks]

In [ ]:
worker = make_agent(
    "worker",
    "Answer the single subtask in one short, factual sentence.",
    max_tokens=8000,
)

def orchestrate(task, verbose=True):
    # 1. MANAGER plans
    subtasks = decompose(task)
    if verbose:
        print("Subtasks the manager chose:")
        for i, st in enumerate(subtasks, 1):
            print(f"  {i}. {st}")

    # 2. WORKERS solve each piece
    findings = [worker(st) for st in subtasks]

    # 3. MANAGER synthesises
    synthesis = ask(
        "Combine these findings into a tidy 2-sentence answer:\n"
        + "\n".join(f"- {f}" for f in findings),
        temperature=0,
        max_tokens=8000,
    )
    return synthesis

In [ ]:
result = orchestrate("Plan a one-day visit to Oslo.")
print("\nSYNTHESIS:\n", result)

> Key idea: the orchestrator-worker pattern trades control for flexibility. You no longer dictate the steps, so the system can handle tasks you didn't anticipate, but you also give up some predictability (the manager might choose a poor breakdown). That trade-off is exactly what the more dynamic control flows in notebook 09 are about.

> Cost. Count the calls: 1 to decompose + 1 per subtask + 1 to synthesise. A 3-subtask job is 5 LLM calls. Decomposition is powerful but not free, so cap the number of subtasks (we used `max_subtasks=3`).

## 4. Parallel fan-out / map-reduce

Look back at the orchestrator's workers: each subtask was independent, yet we ran them one after another in a list comprehension. When pieces don't depend on each other, running them in sequence is just wasted wall-clock time. Every worker is mostly sitting idle waiting for the network.

Fan-out runs the independent workers concurrently (the map), then a final step aggregates their outputs (the reduce).

```
            ┌─▶ [ worker ] ─┐
input ──────┼─▶ [ worker ] ─┼──▶ [ reduce ] ─▶ result
(split)     └─▶ [ worker ] ─┘   (aggregate)
            all at once
```

**When to use it:** many independent items to process the same way, such as summarise 20 documents, get a fact about each of N cities, or score several candidate answers. Anything you'd write as a `for` loop over independent items is a candidate.

![Sequential vs parallel execution on a time axis](figures/08-parallel-speedup.svg)

*Figure: why running independent workers concurrently saves wall-clock time. Generate it from the prompt in [`figures/FIGURES.md`](figures/FIGURES.md).*

Our OpenAI `client` is synchronous, but the work is I/O-bound: each call spends almost all its time waiting on the network, not using the CPU. That is the textbook case where Python threads help: while one thread waits for its response, the others can fire theirs. `concurrent.futures.ThreadPoolExecutor` gives us a tidy `.map()` for exactly this.

First, the map function (one worker, one item) and a small list of items:

In [ ]:
CITIES = ["Oslo", "Bergen", "Tromsø", "Trondheim", "Stavanger"]

def city_fact(city):
    """One worker: 2 short sentences about one city."""
    return ask(f"Give 2 short sentences about {city}, Norway.", max_tokens=8000)

Sequential baseline, a plain loop. Time it so we have something to beat.

In [ ]:
t0 = time.perf_counter()
seq_results = [city_fact(c) for c in CITIES]
t_seq = time.perf_counter() - t0
print(f"sequential: {t_seq:.1f}s for {len(CITIES)} workers")

Parallel version: same work, but every worker is launched at once. `ex.map` returns results in input order, so the aggregation step is unchanged.

In [ ]:
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=len(CITIES)) as ex:
    par_results = list(ex.map(city_fact, CITIES))
t_par = time.perf_counter() - t0

print(f"parallel  : {t_par:.1f}s")
print(f"speedup   : {t_seq / t_par:.1f}× faster\n")
for city, fact in zip(CITIES, par_results):
    print(f"  {city:11} → {fact.splitlines()[0][:60]}")

Now the reduce: collapse the workers' outputs into one result with a single aggregating call.

In [ ]:
summary = ask(
    "Write ONE sentence that captures the common theme of these notes:\n"
    + "\n".join(par_results),
    temperature=0,
    max_tokens=8000,
)
print("REDUCED:", summary)

> Why you care. The fan-out's speedup grows with the number of independent items: 5 workers here, but 50 documents would turn roughly 50x sequential latency into roughly the time of a single call (bounded by `max_workers` and the endpoint's rate limits). For batch-style agent work this is the single biggest, easiest win.

> Rule of thumb: use threads (`ThreadPoolExecutor`) for I/O-bound LLM calls. It is simple and works with the synchronous SDK. If you outgrow it, the production alternative is `asyncio` with an async client (`await` each call, `asyncio.gather(...)` to fan out). Same shape, more scalable; threads are fine to start.

> Gotcha: only fan out independent work. If worker B needs worker A's output, that's a pipeline, not a fan-out, and running them concurrently would just race. And mind the endpoint: too many simultaneous calls can hit rate limits, so cap `max_workers`.

## Patterns compose

These four are not rivals; they're Lego bricks. Real systems nest them:

- A pipeline stage can itself be a router (one stage classifies, then dispatches).
- An orchestrator's workers can run as a parallel fan-out instead of a sequential loop (swap the list comprehension in `orchestrate` for the `ThreadPoolExecutor` block, which is the only change).
- A router can send some categories to a single agent and others to a whole pipeline.

```
[ router ] ─▶ "report" ─▶ [ orchestrator ] ─▶ (fan-out workers) ─▶ [ pipeline: draft ▸ polish ]
```

> Key idea: don't ask "which one pattern is my system?" Ask "what's the data-flow shape of this part?", and compose. The capstone in notebook 10 stitches several of these together.

## Which pattern when?

| Pattern | Shape | Who decides the steps | Use when... | Watch out for |
|---|---|---|---|---|
| Sequential pipeline | A -> B -> C | you (fixed order) | the job has stable, ordered stages (outline -> draft -> polish) | errors propagate down the chain |
| Router / dispatcher | classify -> 1 specialist | a cheap classifier | one request belongs to one of several specialists | ambiguous / missing categories |
| Orchestrator-worker | decompose -> workers -> synthesise | the manager LLM, per task | open-ended tasks with an unknown breakdown | cost (many calls); unpredictable plans |
| Parallel fan-out | map all -> reduce | you (over independent items) | many independent items processed the same way | only for independent work; rate limits |

> Mental model recap. Pipeline = fixed order. Router = pick one. Orchestrator-worker = plan, then delegate. Fan-out = do it all at once, then merge. Everything else is composition.

## Recap

You can now:
- chain agents into a sequential pipeline with a generic `pipeline([...], task)`
- build a router: a JSON-mode, `temperature=0` classifier that dispatches to the right specialist
- run an orchestrator-worker loop: an LLM decomposes a task into JSON subtasks, workers solve them, the manager synthesises
- fan out independent workers concurrently with `ThreadPoolExecutor` and reduce the results, for a real wall-clock speedup
- reason about which pattern fits a piece of work, and compose them

### Try it yourself
1. Add a fourth specialist (e.g. `"translation"`) to the router and route a sentence to it. What category does an ambiguous request get?
2. Rewrite `orchestrate()` so its workers run as a parallel fan-out instead of a sequential list comprehension, then time the difference.
3. Build a 2-stage pipeline whose first stage is a router: stage 1 classifies a question's topic, stage 2 answers it with a topic-specific system prompt.
4. Fan out the `city_fact` worker over 10 cities and watch the speedup grow. Then add `max_workers=2` and explain what changes.

### Next up
09 · Advanced orchestration. The patterns here had static control flow (you, or a one-shot plan, fixed the steps). Next we make the control flow dynamic: a planner-executor that re-plans as it goes, reflection / self-critique where an agent reviews and improves its own output, and a supervisor that decides at each step which agent runs next. Then notebook 10 puts several of these patterns together in the capstone.